#### This notebook is used to check out the capability of analyzing full text papers.

The full text papers may be downlaoded from PMC in PDF or directly placed into a certain folder by curators.

In [54]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('../reactome_llm')
import ReactomeUtils as utils
import ReactomeNeo4jUtils as neo4jutils
import ReactomePrompts as prompts
import importlib
importlib.reload(utils)
importlib.reload(neo4jutils)
importlib.reload(prompts)

from langchain_community.document_loaders import PyPDFLoader

random_state = utils.RANDOM_STATE

#### Load a full paper

In [2]:
pdf_file = '../data/papers/zns15102.pdf'

loader = PyPDFLoader(pdf_file)
token_splitter = utils._get_text_splitter()
pages = loader.load_and_split(token_splitter)

pages[1]

2024-03-04 10:47:27,251 -  sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: pritamdeka/S-PubMedBert-MS-MARCO
2024-03-04 10:47:27,910 -  sentence_transformers.SentenceTransformer - INFO - Use pytorch device: cpu


Document(page_content='in cultured neuronsincreases the density of dendritic spines and excitatory synapses in a manner that requires the pdz ( psd - 95 / dlg / zo - 1 ) - binding c terminiof tanc proteins. tanc1 - deficient mice exhibit reduced spine density in the ca3 region of the hippocampus, but not in the ca1 ordentate gyrus regions, and show impaired spatial memory. tanc2 deficiency, however, causes embryonic lethality. these results suggestthat tanc1 is important for dendritic spine maintenance and spatial memory, and implicate tanc2 in embryonic development. introduction psd - 95 / sap90 ( postsynaptic density - 95 / synapse - associated pro - tein 90 ), an abundant postsynaptic scaffolding protein at excitatorysynapses, has been suggested to regulate dendritic spines and excita - tory synapses through interaction with various postsynaptic densityproteins ( funke et al., 2005 ; montgomery et al., 2004 ; fitzjohn et al., 2006 ; okabe, 2007 ; sheng and hoogenraad, 2007 ; keith a

In [3]:
from langchain_community.vectorstores import FAISS

embeddings = utils._get_embedding()
# The whole embedding took about 15 minutes at the 14'' MacBook Pro (no gpu was used)
db = FAISS.from_documents(pages, embeddings)

2024-03-04 10:47:28,118 -  sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: pritamdeka/S-PubMedBert-MS-MARCO
2024-03-04 10:47:28,243 -  sentence_transformers.SentenceTransformer - INFO - Use pytorch device: cpu


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

2024-03-04 10:47:33,358 -  faiss.loader - INFO - Loading faiss.
2024-03-04 10:47:33,388 -  faiss.loader - INFO - Successfully loaded faiss.


In [4]:
importlib.reload(utils)
importlib.reload(prompts)

query_gene = 'TANC1'
query = f'({query_gene} interactions) or ({query_gene} reactions) or ({query_gene} pathways)'

parameters = {
    'query_gene': query_gene
}
prompt = prompts.relationship_extraction_prompt
model = utils.get_default_llm()

# Pick top 25 
matched_pages = db.similarity_search_with_score(query, k=4)

max_score = 50

for doc, score in matched_pages:
    if score > max_score:
        break
    parameters['docs'] = doc.page_content
    parameters['document'] = doc.page_content
    result = utils.invoke_llm(model=model, parameters=parameters, prompt=prompt)
    print(utils.output_llm_result(result))
    # break

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2024-03-04 10:47:39,586 -  httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Result: TANC1 - expressed_in -> brain (tissue)
TANC1 - localized_at -> excitatory synapses
TANC1 - interacts_with -> PSD-95
TANC1 - forms_complex_with -> GKAP
TANC1 - forms_complex_with -> AMPA receptors
TANC1 - forms_complex_with -> NMDA receptors
TANC1 - forms_complex_with -> Ca2/calmodulin-dependent protein kinase II
TANC1 - interacts_with -> MINK
TANC1 - interacts_with -> TNIK
TANC1 - phosphorylated_by -> MINK
TANC1 - phosphorylated_by -> TNIK
TANC1 - regulated_by -> Rap2

Doc: rolling pebbles regulate myoblast fusion in drosophila ( menon and chia, 2001 ; rau et al., 2001 ). tanc1 mrna in rats is expressed in various tissues, including the brain, where its expression is particularstrong in the hippocampus ( suzuki et al., 2005 ). tanc1 proteinsare localized at excitatory synapses, directly interact with psd - 95, and form a complex with diverse excitatory postsynaptic proteinsincluding psd - 95, gkap, ampa receptors, nmda receptors, and the / h9251subunit of ca2 / h11001 / calmod

2024-03-04 10:47:41,694 -  httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Result: TANC1 - interacts with -> PSD-95, PSD-93, CHAPSYN-110, SAP102
TANC1 - forms a complex with -> TANC2
TANC1 - colocalizes with -> PSD-95 in intracellular clusters
TANC1 - binds to -> PSD-95, PSD-93, SAP97, SAP102
TANC1 - requires the c terminus for interaction with -> PSD-95
TANC1 - spine localization is promoted by -> PSD-95

Doc: in yeast two - hybrid assays, tanc1 interacted with the pdz domains from psd - 95 and psd - 95 relatives ( psd - 93 / chapsyn - 110 and sap102 ), but not with other pdz domains ; these inter - actions were dependent on the c terminus of tanc1 ( fig. 1 b ). in addition, tanc1 and tanc2 formed a complex with psd - 95in a manner that required the last three amino acid residues ( fig. 1c ). in protein coclustering assays, cos cells doubly expressing psd - 95 and tanc2 formed discrete intracellular clusters whereboth proteins colocalized, whereas those expressing psd - 95 anda psd - 95 binding - deficient tanc2 mutant ( tanc2 / h9004c ) did not ( fig. 1 d 

2024-03-04 10:47:43,969 -  httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Result: TANC1 - interacts with PSD-95 family proteins (PSD-95, PSD-93/Chapsyn-110, and SAP102) in yeast two-hybrid assays.
TANC1 - forms a complex with PSD-95 in heterologous cells.
TANC1 - interacts with PSD-95 in a PDZ-dependent manner.
TANC1 - interacts with TANC2 in a complex with PSD-95 in heterologous cells.

Doc: pb, pdz - binding motif that ends with the esnv * sequence. b, tanc1 interacts with the pdz domains from psd - 95 family proteins ( psd - 95, psd - 93 / chapsyn - 110, and sap102 ) but not with those from grip1 ( a control pdz protein ) in yeast two - hybrid assays. pgad10 alone, empty prey vector. hi s3 activity : / h11001 / h11001 / h11001 ( / h1102260 % ), / h11001 / h11001 ( 30 / h1101160 % ), / h11001 ( 10 / h1101130 % ), / h11002 ( no significant growth ) ; / h9252 - gal : / h11001 / h11001 / h11001 ( / h1102145 min ), / h11001 / h11001 ( 45 / h1101190 min ), / h11001 ( 90 / h11011240 min ), / h11002 ( no significant / h9252 - galactosidase activity ). c, tanc1 a

2024-03-04 10:47:47,049 -  httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Result: TANC1 - phosphorylated by MINK/TNIK -> inhibition of TANC1 and suppression of dendritic spines and excitatory synapses
TANC1 - important for spatial memory -> TANC1/H11002 mice show reduced performance in Morris water maze assay
TANC1 - not important for locomotive exploration, motor coordination, or basal anxiety-like behaviors -> TANC1/H11002 mice exhibit normal behaviors in open-field, rotarod assays, and elevated plus maze
TANC1 - involved in learned anxiety -> TANC1/H11002 mice show normal behaviors in open-field, elevated plus maze, and rotarod assays

Doc: tanc1 phosphorylation by mink / tnik induced by rap2activation might result in inhibition of tanc1 and suppression ofdendritic spines and excitatory synapses. clues for the functions of tanc proteins at the organismal level come from the phenotypes of tanc1 / h11002 / / h11002and tanc2 / h11002 / / h11002 mice. tanc1 / h11002 / / h11002mice show reduced performance in the morris water maze assay, while exhibiting norm

#### Wrap everything into a function in the utils script

In [8]:
from ast import mod


importlib.reload(utils)

full_text_results = utils.analyze_full_paper(paper_file_name=pdf_file, query_gene=query_gene, model = model)

2024-03-05 17:01:18,863 -  sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: pritamdeka/S-PubMedBert-MS-MARCO
2024-03-05 17:01:19,352 -  sentence_transformers.SentenceTransformer - INFO - Use pytorch device: cpu
2024-03-05 17:01:19,561 -  sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: pritamdeka/S-PubMedBert-MS-MARCO
2024-03-05 17:01:19,706 -  sentence_transformers.SentenceTransformer - INFO - Use pytorch device: cpu


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2024-03-05 17:01:27,452 -  httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2024-03-05 17:01:28,697 -  httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2024-03-05 17:01:30,850 -  httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2024-03-05 17:01:34,395 -  httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2024-03-05 17:01:35,755 -  httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2024-03-05 17:01:37,867 -  httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2024-03-05 17:01:40,076 -  httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2024-03-05 17:01:42,933 -  httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2024-03-05 17:01:45,295 

In [13]:
full_test_result = full_text_results[0]
full_test_result

{'answer': AIMessage(content='TANC1 - expressed_in -> brain (tissue)\nTANC1 - localized_at -> excitatory synapses\nTANC1 - interacts_with -> PSD-95\nTANC1 - interacts_with -> GKAP\nTANC1 - interacts_with -> AMPA receptors\nTANC1 - interacts_with -> NMDA receptors\nTANC1 - interacts_with -> H9251 subunit of Ca2/calmodulin-dependent protein kinase II\nTANC1 - phosphorylated_by -> MINK\nTANC1 - phosphorylated_by -> TNIK\nTANC1 - regulated_by -> Rap2'),
 'docs': 'rolling pebbles regulate myoblast fusion in drosophila ( menon and chia, 2001 ; rau et al., 2001 ). tanc1 mrna in rats is expressed in various tissues, including the brain, where its expression is particularstrong in the hippocampus ( suzuki et al., 2005 ). tanc1 proteinsare localized at excitatory synapses, directly interact with psd - 95, and form a complex with diverse excitatory postsynaptic proteinsincluding psd - 95, gkap, ampa receptors, nmda receptors, and the / h9251subunit of ca2 / h11001 / calmodulin - dependent protein

In [9]:
for full_text_result in full_text_results:
    print(utils.output_llm_result(full_text_result))


Result: TANC1 - expressed_in -> brain (tissue)
TANC1 - localized_at -> excitatory synapses
TANC1 - interacts_with -> PSD-95
TANC1 - interacts_with -> GKAP
TANC1 - interacts_with -> AMPA receptors
TANC1 - interacts_with -> NMDA receptors
TANC1 - interacts_with -> H9251 subunit of Ca2/calmodulin-dependent protein kinase II
TANC1 - phosphorylated_by -> MINK
TANC1 - phosphorylated_by -> TNIK
TANC1 - regulated_by -> Rap2

Doc: rolling pebbles regulate myoblast fusion in drosophila ( menon and chia, 2001 ; rau et al., 2001 ). tanc1 mrna in rats is expressed in various tissues, including the brain, where its expression is particularstrong in the hippocampus ( suzuki et al., 2005 ). tanc1 proteinsare localized at excitatory synapses, directly interact with psd - 95, and form a complex with diverse excitatory postsynaptic proteinsincluding psd - 95, gkap, ampa receptors, nmda receptors, and the / h9251subunit of ca2 / h11001 / calmodulin - dependent protein kinase ii. more recently, tanc1 was 

#### Code for downloading a PDF from PMC based on PMID

In [57]:
from pathlib import Path
from metapub import FindIt
import ReactomeLLMErrors as errors
import requests

def find_paper_download(pmid: str, dir_path: str) -> str:
    """Find a full text pdf paper and then download it. It may not success.

    Args:
        pmid (str): _description_
        dir_path (str): _description_

    Returns:
        str: _description_
    """
    # Find the URL first
    src = FindIt(pmid=pmid)
    if src is None:
        raise errors.PubMedFullTextPDFNotFoundError(pmid)
    # Start to download
    # src.url = 'https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3206016/pdf/pone.0025376.pdf'
    response = requests.get(src.url)
    file_name = Path(dir_path, '{}.pdf'.format(pmid))
    with open(file_name, 'wb') as file:
        file.write(response.content)
    

# url = FindIt('22069443')
# print(url.url)
pmid = 22069443
dir_path = '../data/papers'
find_paper_download(pmid, dir_path)

In [59]:
file_path = Path("/Users/wug/git/reactome/curator-tool-llm/data/papers/28754924.pdf")
file_path.exists()

False